# KaBuM — Feature Engineering
Extrai atributos técnicos do campo `nome` de cada categoria usando expressões regulares.

**Categorias:** CPU · GPU · Placa-mãe · SSD · Fonte · RAM

In [1]:
import re
import pathlib
import pandas as pd

pd.set_option("display.max_colwidth", 80)

## 1. Carregar dados
Ajuste `DATA_DIR` e `DATA_COLETA` para a pasta e data da sua coleta.

In [2]:
DATA_DIR    = pathlib.Path(r"C:\Users\julia\OneDrive\Área de Trabalho\Projetos\Perspectivas de Dados\perspectiva-dado\Projetos\Projeto 1\00_Dados")
DATA_COLETA = "2026-05-16"  # ajuste para a data da sua coleta

pasta = DATA_DIR / DATA_COLETA

categorias = ["cpu", "gpu", "placa_mae", "ssd", "fonte", "ram"]

dfs = {}
for cat in categorias:
    arquivo = pasta / f"kabum_{cat}_{DATA_COLETA}.csv"
    dfs[cat] = pd.read_csv(arquivo, encoding="utf-8-sig")
    print(f"{cat:10s}: {len(dfs[cat])} produtos")

cpu       : 482 produtos
gpu       : 629 produtos
placa_mae : 842 produtos
ssd       : 867 produtos
fonte     : 616 produtos
ram       : 1200 produtos


## 2. Funções auxiliares

In [3]:
def extrair(nome, padrao, tipo=str, grupo=1):
    """
    Aplica um regex ao campo nome e retorna o grupo capturado.
    Retorna None se não encontrar.
    """
    if pd.isna(nome):
        return None
    match = re.search(padrao, nome, re.IGNORECASE)
    if match:
        try:
            return tipo(match.group(grupo))
        except (ValueError, IndexError):
            return None
    return None


def contem(nome, padrao):
    """Retorna True se o padrão for encontrado no nome."""
    if pd.isna(nome):
        return False
    return bool(re.search(padrao, nome, re.IGNORECASE))

## 3. RAM

In [4]:
def features_ram(df):
    d = df.copy()
    n = d["nome"]

    d["ram_geracao"]    = n.apply(lambda x: extrair(x, r"(DDR[2-5])"))
    d["ram_gb"]         = n.apply(lambda x: extrair(x, r"(\d+)\s*GB", int))
    d["ram_mhz"]        = n.apply(lambda x: extrair(x, r"(\d{3,5})\s*MHz", int))
    d["ram_cl"]         = n.apply(lambda x: extrair(x, r"CL(\d+)", int))
    d["ram_notebook"]   = n.apply(lambda x: contem(x, r"Notebook|SODIMM|SO-DIMM"))

    return d

dfs["ram"] = features_ram(dfs["ram"])

cols = ["nome", "ram_geracao", "ram_gb", "ram_mhz", "ram_cl", "ram_notebook"]
display(dfs["ram"][cols].head(10))
print("\nValores nulos por feature:")
print(dfs["ram"][cols[1:]].isna().sum())

,nome,ram_geracao,ram_gb,ram_mhz,ram_cl,ram_notebook
0,"Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4, CL22, Preto - HRM001083222PT",DDR4,8.0,3200.0,22.0,False
1,"Memória RAM Kingston Fury Beast, 16GB, 3200MHz, DDR4, CL16, Preto - KF432C16...",DDR4,16.0,3200.0,16.0,False
2,"Memória RAM Rise Mode Z, 8GB, 3200MHz, DDR4, CL19, Preto - RM-D4-8G-3200Z",DDR4,8.0,3200.0,19.0,False
3,"Memória para Notebook Rise Mode Value, 8GB, 3200MHz, DDR4, CL22, Preto - RM-...",DDR4,8.0,3200.0,22.0,True
4,"Memória RAM Kingston Fury Beast, 8GB, 3200MHz, DDR4, CL16, Preto - KF432C16BB/8",DDR4,8.0,3200.0,16.0,False
5,"Memoria RAM Para Notebook Rise Mode Value, 16GB, 3200MHZ, DDR4, CL22 - RM-D4...",DDR4,16.0,3200.0,22.0,True
6,"Memória RAM XPG Lancer, RGB, 32GB (2x16GB), 6000MHz, DDR5, CL30, Preto - AX5...",DDR5,32.0,6000.0,30.0,False
7,"Memória RAM Husky Impulse, 16GB, 3200MHz, DDR4, CL22, Preto - HRM001163222PT",DDR4,16.0,3200.0,22.0,False
8,"Memória RAM Rise Mode, 8GB, 1600MHz, DDR3, CL11 - RM-D3-8G1600V",DDR3,8.0,1600.0,11.0,False
9,"Memória RAM para Notebook Rise Mode, 8GB, 2400MHz, DDR4, CL17 - RM-D4-8G-2400VN",DDR4,8.0,2400.0,17.0,True



Valores nulos por feature:
ram_geracao      88
ram_gb            7
ram_mhz         330
ram_cl          867
ram_notebook      0
dtype: int64


## 4. CPU

In [20]:
# Tabela de referência: socket → DDR suportado
# (usada quando o nome não informa a geração DDR)
SOCKET_DDR = {
    "AM5":     "DDR5",
    "AM4":     "DDR4",
    "AM3":     "DDR3",
    "AM2":     "DDR2",
    "FM2":     "DDR3",
    "STR5":    "DDR5",
    "SP3":     "DDR4",
    "LGA1851": "DDR5",
    "LGA1700": "DDR4/DDR5",
    "LGA1200": "DDR4",
    "LGA1151": "DDR4",
    "LGA1150": "DDR3",
    "LGA1155": "DDR3",
    "LGA2011": "DDR3/DDR4",
}

def features_cpu(df):
    d = df.copy()
    n = d["nome"]

    d["cpu_socket"] = n.apply(lambda x: extrair(x, 
        r"(AM[2-5]|FM[12]|STR5|SP3"           # AMD
        r"|LGA\s*\d{3,4}"                      # LGA com prefixo
        r"|Sk(\d{3,4})"                        # Sk1151, Sk1700
        r"|(\d{3,4})[pP]\b"                    # 1151p, 1200p
        r"|L(\d{4})\b"                         # L1700
        r"|\((\d{4})\)"                        # (1851)
        r"|\b(1[0-9]\d{2}|20[0-9]{2})\s+(?:Intel|Core|i[3579]|Xeon)"  # 1700 Intel
        r")"
    ))

    def normalizar_socket(s):
        if pd.isna(s):
            return None
        s = re.sub(r"\s+", "", s).upper()
        # Remove Sk, L, p, parênteses — fica só o número
        num = re.search(r"\d{3,4}", s)
        if not num:
            return s
        n = num.group()
        # Sockets que não são LGA
        if s in ("AM2", "AM3", "AM4", "AM5", "FM1", "FM2", "STR5", "SP3"):
            return s
        # Adiciona LGA nos numéricos
        return f"LGA{n}"

    d["cpu_socket"] = d["cpu_socket"].apply(normalizar_socket)

    d["cpu_tdp_w"]   = n.apply(lambda x: extrair(x, r"(\d+)\s*W(?!h)", int))
    d["cpu_com_cooler"] = n.apply(lambda x: contem(x, r"Box|Wraith|Cooler|Fan"))
    d["cpu_marca"]   = n.apply(lambda x:
        "Intel" if contem(x, r"Intel|Core\s*i[3579]|Core\s*Ultra") else
        "AMD"   if contem(x, r"AMD|Ryzen|Athlon") else None
    )
    d["cpu_serie"]   = n.apply(lambda x: extrair(x, r"(Ryzen\s*[3579]|Core\s*i[3579]|Core\s*Ultra\s*\d)"))

    # DDR suportado via tabela de referência
    d["cpu_ddr_suportado"] = d["cpu_socket"].map(SOCKET_DDR)

    return d

dfs["cpu"] = features_cpu(dfs["cpu"])

cols = ["nome", "cpu_marca", "cpu_socket", "cpu_serie", "cpu_tdp_w", "cpu_ddr_suportado", "cpu_com_cooler"]
display(dfs["cpu"][cols].head(10))
print("\nValores nulos por feature:")
print(dfs["cpu"][cols[1:]].isna().sum())

,nome,cpu_marca,cpu_socket,cpu_serie,cpu_tdp_w,cpu_ddr_suportado,cpu_com_cooler
0,"Processador AMD Ryzen 5 5600GT, 3.6 GHz, (4.6GHz Max Turbo), Cache 4MB, 6 Nú...",AMD,AM4,Ryzen 5,NaN,DDR4,True
1,"Processador AMD Ryzen 5 5500, 3.6GHz (4.2GHz Max Turbo), Cache 19MB, AM4, Se...",AMD,AM4,Ryzen 5,NaN,DDR4,True
2,"Processador AMD Ryzen 7 5700X, 3.4GHz (4.6GHz Max Turbo), Cache 36MB, AM4, S...",AMD,AM4,Ryzen 7,100000926.0,DDR4,False
3,"Processador AMD Ryzen 7 7800X3D, 5.0GHz Max Turbo, Cache 104MB, AM5, 8 Núcle...",AMD,AM5,Ryzen 7,100000910.0,DDR5,False
4,"Processador AMD Ryzen 7 9800X3D, Cache 8MB, 8 Núcleos, 16 Threads, AM5 - 100...",AMD,AM5,Ryzen 7,100001084.0,DDR5,False
5,"Processador AMD Ryzen 5 5500, 3.6GHz, Cache 16MB, Hexa Core, 12 Threads, AM4...",AMD,AM4,Ryzen 5,NaN,DDR4,True
6,"Processador AMD Ryzen 3 3200G, 3.6GHz (4GHz Max Turbo), Cache 4MB, Quad Core...",AMD,AM4,Ryzen 3,NaN,DDR4,True
7,"Processador AMD Ryzen 9 9950x3d, 4,4 GHz, (Máx Boos Clock Até 5,5 GHz), Cach...",AMD,AM5,Ryzen 9,100000719.0,DDR5,False
8,"Processador AMD Ryzen 5 7600X, 5.3GHz Max Turbo, Cache 38MB, AM5, 6 Núcleos...",AMD,AM5,Ryzen 5,100000593.0,DDR5,False
9,"Processador AMD Ryzen 7 5700G, 3.8GHz (4.6GHz Max Turbo), Cache 20MB, 8 Núcl...",AMD,AM4,Ryzen 7,NaN,DDR4,True



Valores nulos por feature:
cpu_marca              2
cpu_socket            75
cpu_serie             94
cpu_tdp_w            433
cpu_ddr_suportado     77
cpu_com_cooler         0
dtype: int64


## 5. Placa-mãe

In [7]:
def features_placa_mae(df):
    d = df.copy()
    n = d["nome"]

    d["mobo_socket"]     = n.apply(lambda x: extrair(x, r"(AM[45]|LGA\s?\d{3,4})"))
    d["mobo_socket"]     = d["mobo_socket"].str.replace(" ", "").str.upper()
    d["mobo_chipset"]    = n.apply(lambda x: extrair(x, r"\b([A-Z]\d{3,4})(?!\s*MHz)\b"))
    d["mobo_ddr"]        = n.apply(lambda x: extrair(x, r"(DDR[45])"))
    d["mobo_form_factor"]= n.apply(lambda x: extrair(x, r"\b(ATX|mATX|m-ATX|Micro-ATX|Mini-ATX|ITX|Mini-ITX)\b"))
    d["mobo_slots_m2"]   = n.apply(lambda x: extrair(x, r"(\d+)\s*x?\s*M\.2", int))
    d["mobo_max_ram_gb"] = n.apply(lambda x: extrair(x, r"(\d+)\s*GB(?=.*RAM)", int))

    return d

dfs["placa_mae"] = features_placa_mae(dfs["placa_mae"])

cols = ["nome", "mobo_socket", "mobo_chipset", "mobo_ddr", "mobo_form_factor", "mobo_slots_m2"]
display(dfs["placa_mae"][cols].head(10))
print("\nValores nulos por feature:")
print(dfs["placa_mae"][cols[1:]].isna().sum())

,nome,mobo_socket,mobo_chipset,mobo_ddr,mobo_form_factor,mobo_slots_m2
0,"Placa-Mãe MSI A520M-A PRO, AMD AM4, mATX, DDR4, Preto - A520M-A PRO",AM4,None,DDR4,mATX,NaN
1,"Placa-Mãe MSI MPG B550 Gaming Plus, AMD AM4, ATX, DDR4, Preto - MPG B550 GAM...",AM4,B550,DDR4,ATX,NaN
2,"Placa-Mãe Gigabyte B550M Aorus Elite Rev. 1.3, AMD AM4, Micro ATX, DDR4, Pre...",AM4,None,DDR4,ATX,NaN
3,"Placa-Mãe ASUS TUF GAMING A520M-PLUS II, AMD AM4, mATX, DDR4, Preto - 90MB17...",AM4,None,DDR4,mATX,NaN
4,"Placa-Mãe ASUS TUF Gaming B550M-Plus, AMD AM4, mATX, DDR4, Preto - 90MB14A0-...",AM4,None,DDR4,mATX,NaN
5,"Placa-Mãe MSI B550M Pro-VDH WiFi, AMD AM4, mATX, DDR4, Preto - B550M PRO-VDH...",AM4,None,DDR4,mATX,NaN
6,"Placa Mãe Gigabyte B550M DS3H AC R2, AMD AM4, Micro ATX, DDR4, RGB, Wi-Fi, B...",AM4,None,DDR4,ATX,NaN
7,"Placa-Mãe MSI PRO B760M-P, Intel B760, DDR4, Preto - PRO B760M-P DDR4",None,B760,DDR4,None,NaN
8,"Placa-Mãe Asus TUF Gaming B550M-PLUS, AMD AM4, mATX , DDR4, M.2, Aura para f...",AM4,None,DDR4,mATX,NaN
9,"Placa Mãe ASUS B650M-AYW Wi-Fi, AM5, mATX, DDR5, Wi-fi, Bluetooth - 90MB1KI0...",AM5,None,DDR5,mATX,NaN



Valores nulos por feature:
mobo_socket         348
mobo_chipset        464
mobo_ddr            230
mobo_form_factor    342
mobo_slots_m2       816
dtype: int64


## 6. GPU

In [8]:
# Tabela de referência: modelo GPU → TDP aproximado (Watts)
GPU_TDP = {
    "RTX 5090": 575, "RTX 5080": 360, "RTX 5070 Ti": 300, "RTX 5070": 250,
    "RTX 5060 Ti": 180, "RTX 5060": 150,
    "RTX 4090": 450, "RTX 4080": 320, "RTX 4070 Ti": 285, "RTX 4070": 200,
    "RTX 4060 Ti": 165, "RTX 4060": 115,
    "RTX 3090": 350, "RTX 3080": 320, "RTX 3070": 220, "RTX 3060": 170,
    "RX 9070 XT": 304, "RX 9070": 220,
    "RX 7900 XTX": 355, "RX 7900 XT": 315, "RX 7800 XT": 263, "RX 7700 XT": 245,
    "RX 7600": 165, "RX 6700 XT": 230, "RX 6600": 132,
}

def extrair_modelo_gpu(nome):
    """Retorna o modelo GPU mais longo encontrado no nome."""
    if pd.isna(nome):
        return None
    for modelo in sorted(GPU_TDP.keys(), key=len, reverse=True):
        if re.search(re.escape(modelo), nome, re.IGNORECASE):
            return modelo
    # fallback: captura padrão genérico
    m = re.search(r"(RTX\s?\d{4}(?:\s?Ti)?|RX\s?\d{4}(?:\s?XT)?)", nome, re.IGNORECASE)
    return m.group(1).strip() if m else None

def features_gpu(df):
    d = df.copy()
    n = d["nome"]

    d["gpu_marca_chip"] = n.apply(lambda x:
        "NVIDIA" if contem(x, r"RTX|GTX|NVIDIA") else
        "AMD"    if contem(x, r"\bRX\b|Radeon|AMD") else
        "Intel"  if contem(x, r"Arc|Intel") else None
    )
    d["gpu_modelo"]     = n.apply(extrair_modelo_gpu)
    d["gpu_vram_gb"]    = n.apply(lambda x: extrair(x, r"(\d+)\s*GB", int))
    d["gpu_tdp_w"]      = d["gpu_modelo"].map(GPU_TDP)

    return d

dfs["gpu"] = features_gpu(dfs["gpu"])

cols = ["nome", "gpu_marca_chip", "gpu_modelo", "gpu_vram_gb", "gpu_tdp_w"]
display(dfs["gpu"][cols].head(10))
print("\nValores nulos por feature:")
print(dfs["gpu"][cols[1:]].isna().sum())

,nome,gpu_marca_chip,gpu_modelo,gpu_vram_gb,gpu_tdp_w
0,"Placa de Vídeo RX 7600 GAMING OC 8G AMD Radeon Gigabyte, 8GB, GDDR6, 128bits...",AMD,RX 7600,8.0,165.0
1,"Placa de Vídeo XFX AMD RADEON RX 7600 Gaming Graphics Card, 8GB, GDDR6 - RX-...",AMD,RX 7600,8.0,165.0
2,"Placa De Vídeo Husky Alpha RX 580, AMD, 8GB, GDDR5, 256-Bit, HDMI, DVI, Disp...",AMD,None,8.0,NaN
3,"Placa De Vídeo MSI GeForce RTX 5060 Ti Shadow 2X OC Plus, Nvidia, 8GB, GDDR7...",NVIDIA,RTX 5060 Ti,8.0,180.0
4,"Placa de Vídeo MSI GeForce RTX 5070 12G VENTUS 2X OC,12 GB GDDR7, 28Gbps, NV...",NVIDIA,RTX 5070,12.0,250.0
5,"Placa de Vídeo RX 7600 Series Graphics Cards XFX AMD Radeon, 8GB GDDR6 - RX-...",AMD,RX 7600,8.0,165.0
6,"Placa de Vídeo PowerColor RX 9060 XT Reaper AMD Radeon, 16GB, GDDR6, 128bits...",AMD,RX 9060 XT,16.0,NaN
7,"Placa de Vídeo ASUS RTX 5060 TI DUAL O8G NVIDIA GeForce, 8GB, GDDR7, DLSS, R...",NVIDIA,RTX 5060 Ti,8.0,180.0
8,"Placa de Vídeo MSI GeForce RTX 5050 8G VENTUS 2X OC NVIDIA GeForce, 8GB GDDR...",NVIDIA,RTX 5050,8.0,NaN
9,"Placa de Vídeo ASUS Dual RX 9060 XT AMD Radeon, 16GB, GDDR6 - DUAL-RX9060XT-16G",AMD,RX 9060 XT,16.0,NaN



Valores nulos por feature:
gpu_marca_chip     94
gpu_modelo        222
gpu_vram_gb        50
gpu_tdp_w         442
dtype: int64


## 7. SSD

In [9]:
def features_ssd(df):
    d = df.copy()
    n = d["nome"]

    d["ssd_interface"]  = n.apply(lambda x:
        "NVMe" if contem(x, r"NVMe|M\.2") else
        "SATA" if contem(x, r"SATA") else None
    )
    d["ssd_geracao_pcie"]= n.apply(lambda x: extrair(x, r"PCIe\s?(\d\.\d)|Gen\s?(\d)", grupo=1))
    d["ssd_capacidade_gb"] = n.apply(lambda x: (
        extrair(x, r"(\d+)\s*TB", float) * 1024
        if contem(x, r"\d+\s*TB") else
        extrair(x, r"(\d+)\s*GB", int)
    ))
    d["ssd_notebook"]   = n.apply(lambda x: contem(x, r"Notebook|2230|2242"))

    return d

dfs["ssd"] = features_ssd(dfs["ssd"])

cols = ["nome", "ssd_interface", "ssd_geracao_pcie", "ssd_capacidade_gb", "ssd_notebook"]
display(dfs["ssd"][cols].head(10))
print("\nValores nulos por feature:")
print(dfs["ssd"][cols[1:]].isna().sum())

,nome,ssd_interface,ssd_geracao_pcie,ssd_capacidade_gb,ssd_notebook
0,"SSD Kingston NV3, 1 TB, M.2 2280, PCIe 4.0 x4, NVMe, Leitura: 6000 MB/s, Gra...",NVMe,4.0,1024.0,False
1,"SSD Rise Mode Gamer Line, 120GB, SATA III, Leitura: 530MB/s, Gravação: 520MB...",SATA,None,120.0,False
2,"SSD Kingston A400, 480GB, SATA III, 2.5"", Leitura: 500MB/s, Gravação: 450MB/...",SATA,None,480.0,False
3,"SSD Husky, SATA III, 256GB, 2.5"", Leitura 500MB/s, Gravação 450MB/s, Preto -...",SATA,None,256.0,False
4,"SSD Husky 128GB, SATA III, 2.5"", Leitura 500MB/s, Gravação 450MB/s, Preto - ...",SATA,None,128.0,False
5,"SSD Rise Mode Gamer Line, 256GB, SATA III, Leitura: 530MB/s, Gravação: 520MB...",SATA,None,256.0,False
6,"SSD Kingston A400, 960GB, SATA III, 2.5"", Leitura: 500MB/s, Gravação: 450MB/...",SATA,None,960.0,False
7,"SSD Msi Spatium S270, 240GB, Sata Iii 2,5 Polegadas - S78-440n070-p83",SATA,None,240.0,False
8,"SSD Kingston NV3, 500 GB, M.2 2280, PCIe 4.0 x4, NVMe, Leitura: 5000 MB/s, G...",NVMe,4.0,500.0,False
9,"SSD Kingston A400, 240GB, SATA III, 2.5"", Leitura: 500MB/s, Gravação: 350MB/...",SATA,None,240.0,False



Valores nulos por feature:
ssd_interface        162
ssd_geracao_pcie     677
ssd_capacidade_gb     83
ssd_notebook           0
dtype: int64


## 8. Fonte

In [10]:
def features_fonte(df):
    d = df.copy()
    n = d["nome"]

    d["fonte_wattagem"]    = n.apply(lambda x: extrair(x, r"(\d{3,4})\s*W(?!h)", int))
    d["fonte_certificacao"]= n.apply(lambda x: extrair(x, r"(Titanium|Platinum|Gold|Silver|Bronze)"))
    d["fonte_modular"]     = n.apply(lambda x:
        "Full"  if contem(x, r"Full.?Modular|Modular\s*Full") else
        "Semi"  if contem(x, r"Semi.?Modular|Modular\s*Semi") else
        "Não"   if contem(x, r"Não.?Modular|Non.?Modular") else None
    )
    d["fonte_atx3"]        = n.apply(lambda x: contem(x, r"ATX\s*3\.0|ATX3"))

    return d

dfs["fonte"] = features_fonte(dfs["fonte"])

cols = ["nome", "fonte_wattagem", "fonte_certificacao", "fonte_modular", "fonte_atx3"]
display(dfs["fonte"][cols].head(10))
print("\nValores nulos por feature:")
print(dfs["fonte"][cols[1:]].isna().sum())

,nome,fonte_wattagem,fonte_certificacao,fonte_modular,fonte_atx3
0,"Fonte MSI MAG A650BN, 650W, 80 Plus Bronze, PFC Ativo, Com Cabo, Preto - 306...",650.0,Bronze,None,False
1,"Fonte Gigabyte P650G PG5, 650W, 80 PLUS Gold, PFC Ativo, Preto - 28200-P65G5...",650.0,Gold,None,False
2,"Fonte Corsair CX Series CX650, 650W, 80 Plus Bronze, Com Cabo, Preto - CP-90...",650.0,Bronze,None,False
3,"Fonte XPG Core Reactor II VE, 850W, 80 Plus Gold, Modular, PFC Ativo, Preto ...",850.0,Gold,None,False
4,"Fonte MSI MAG A600DN, 600W, 80 Plus White, PFC Ativo, Com Cabo, Preto - 306-...",600.0,None,None,False
5,"Fonte MACH1 Steady, 500W, 80 Plus Bronze, PFC Ativo, Preto - 500Steady",500.0,Bronze,None,False
6,"Fonte Gamer Rise Mode Zeus, 500W, PFC Ativo, Preto - RM-PSU-01-WT-500",500.0,None,None,False
7,"Fonte MACH1 Steady, 650W, 80 Plus Bronze, PFC Ativo, Preto - 650Steady",650.0,Bronze,None,False
8,"Fonte Husky Sledger 650W, 80 Plus Bronze, Cybenetics Bronze, PFC ativo, Bivo...",650.0,Bronze,None,False
9,"Fonte MACH1 Steady, 750W, 80 Plus Bronze, PFC Ativo, Preto - 750Steady",750.0,Bronze,None,False



Valores nulos por feature:
fonte_wattagem         30
fonte_certificacao    305
fonte_modular         526
fonte_atx3              0
dtype: int64


## 9. Consolidar e salvar

In [16]:
pasta_saida = DATA_DIR / DATA_COLETA

# Salva cada categoria individualmente com as novas features
for cat, df in dfs.items():
    arquivo = pasta_saida / f"kabum_{cat}_{DATA_COLETA}_features.csv"
    df.to_csv(arquivo, index=False, encoding="utf-8-sig")
    print(f"Salvo: {arquivo.name}")

print("\n✓ Feature engineering concluído!")

Salvo: kabum_cpu_2026-05-16_features.csv
Salvo: kabum_gpu_2026-05-16_features.csv
Salvo: kabum_placa_mae_2026-05-16_features.csv
Salvo: kabum_ssd_2026-05-16_features.csv
Salvo: kabum_fonte_2026-05-16_features.csv
Salvo: kabum_ram_2026-05-16_features.csv

✓ Feature engineering concluído!


## 10. Resumo de cobertura
Verifica quantos produtos tiveram cada feature extraída com sucesso.

In [21]:
features_por_cat = {
    "ram":       ["ram_geracao", "ram_gb", "ram_mhz", "ram_cl"],
    "cpu":       ["cpu_marca", "cpu_socket", "cpu_serie", "cpu_ddr_suportado"],
    "placa_mae": ["mobo_socket", "mobo_chipset", "mobo_ddr", "mobo_form_factor"],
    "gpu":       ["gpu_marca_chip", "gpu_modelo", "gpu_vram_gb", "gpu_tdp_w"],
    "ssd":       ["ssd_interface", "ssd_capacidade_gb"],
    "fonte":     ["fonte_wattagem", "fonte_certificacao", "fonte_modular"],
}

for cat, features in features_por_cat.items():
    df = dfs[cat]
    total = len(df)
    print(f"\n{cat.upper()} ({total} produtos)")
    for f in features:
        preenchidos = df[f].notna().sum()
        pct = preenchidos / total * 100
        barra = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {f:<25s} {barra} {pct:5.1f}% ({preenchidos}/{total})")


RAM (1200 produtos)
  ram_geracao               ██████████████████░░  92.7% (1112/1200)
  ram_gb                    ███████████████████░  99.4% (1193/1200)
  ram_mhz                   ██████████████░░░░░░  72.5% (870/1200)
  ram_cl                    █████░░░░░░░░░░░░░░░  27.8% (333/1200)

CPU (482 produtos)
  cpu_marca                 ███████████████████░  99.6% (480/482)
  cpu_socket                ████████████████░░░░  84.4% (407/482)
  cpu_serie                 ████████████████░░░░  80.5% (388/482)
  cpu_ddr_suportado         ████████████████░░░░  84.0% (405/482)

PLACA_MAE (842 produtos)
  mobo_socket               ███████████░░░░░░░░░  58.7% (494/842)
  mobo_chipset              ████████░░░░░░░░░░░░  44.9% (378/842)
  mobo_ddr                  ██████████████░░░░░░  72.7% (612/842)
  mobo_form_factor          ███████████░░░░░░░░░  59.4% (500/842)

GPU (629 produtos)
  gpu_marca_chip            █████████████████░░░  85.1% (535/629)
  gpu_modelo                ████████████░░░░░░░░ 

In [18]:
sem_socket = dfs["cpu"][dfs["cpu"]["cpu_socket"].isna()]["nome"]
print(sem_socket.tolist())

['Processador AMD Ryzen 5 5600XT, 3.7GHz (4.7GHz Max Turbo), Cache 32MB, 6 Núleos - 100-100001585BOX', 'Processador Intel Core i7-14700F, até 5,40 GHz, Cache 33MB,-Núcleos 20, Threads 28 - BX8071514700F', 'Processador Intel Core Ultra 5-245KF, 5.2GHz, Até 14 Núcleos, Com suporte a PCIe 5.0 e 4.0 e suporte a DDR5 - BX80768245KF', 'Processador Intel Core i3-14100F, até 4,70 GHz, Cache 12MB,-Núcleos 4, Threads 8 - BX8071514100F', 'Processador Intel Core Ultra 9-285K, 5.7GHz, Até 24 Núcleos, Com suporte a PCIe 5.0 e 4.0 e suporte a DDR5 - BX80768285K', 'Processador Intel Core Ultra 7-265KF, 5.5GHz, Até 20 Núcleos, Com suporte a PCIe 5.0 e 4.0 e suporte a DDR5 - BX80768265KF', 'Processador Intel Core Ultra 5 Desktop Processor 225F - BX80768225F', 'Processador Intel Core Ultra 5-245K, 5.2GHz, Até 14 Núcleos, Com suporte a PCIe 5.0 e 4.0 e suporte a DDR5- BX80768245K', 'Processador Intel Core Ultra 5 Desktop Processor 225 - BX80768225', 'Processador Intel Core i9-14900F, até 5,80 GHz, Cache 3